# Proyecto 1 · Ingesta y diagnóstico inicial

**Curso:** CC3084 · Data Science  
**Alcance de este cuaderno:** unir los archivos crudos y diagnosticar su
estado inicial.

> En este cuaderno no se corrigen, normalizan ni eliminan datos.

## 1. Dependencias y rutas

In [ ]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RAW_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")
REPORT_DIR = Path("../reports/diagnostico_inicial")

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Validación de la ingesta y unión de los 23 CSV

In [ ]:
archivos = sorted(RAW_DIR.glob("*.csv"))

print("Archivos encontrados:", len(archivos))
display(pd.DataFrame({"archivo": [a.name for a in archivos]}))

if len(archivos) != 23:
    raise ValueError(
        f"Se esperaban 23 archivos y se encontraron {len(archivos)}."
    )

In [ ]:
dataframes = []
esquemas = []

for archivo in archivos:
    df_archivo = pd.read_csv(
        archivo,
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    esquemas.append({
        "archivo": archivo.name,
        "registros": len(df_archivo),
        "variables": len(df_archivo.columns),
        "columnas": tuple(df_archivo.columns),
    })

    df_archivo["archivo_origen"] = archivo.name
    df_archivo["fila_origen"] = range(2, len(df_archivo) + 2)
    dataframes.append(df_archivo)

resumen_archivos = pd.DataFrame(esquemas)
display(resumen_archivos[["archivo", "registros", "variables"]])

columnas_referencia = esquemas[0]["columnas"]
resumen_archivos["esquema_consistente"] = (
    resumen_archivos["columnas"] == columnas_referencia
)

print(
    "Todos los archivos tienen el mismo esquema:",
    resumen_archivos["esquema_consistente"].all(),
)

In [ ]:
df_raw = pd.concat(dataframes, ignore_index=True)

ruta_unificada = (
    INTERIM_DIR
    / "establecimientos_diversificado_raw_unificado.csv"
)

df_raw.to_csv(
    ruta_unificada,
    index=False,
    encoding="utf-8-sig",
)

print("Dimensión del conjunto unido:", df_raw.shape)
print("Archivo creado:", ruta_unificada)
display(df_raw.head())

### Análisis de la ingesta

Escriba aquí la conclusión sobre:

- cantidad de archivos;
- consistencia de columnas;
- número total de registros;
- confirmación de que los archivos originales permanecen intactos.

## 3.a Número de registros y variables

In [ ]:
columnas_originales = [
    columna for columna in df_raw.columns
    if columna not in ["archivo_origen", "fila_origen"]
]

filas_completamente_vacias = (
    df_raw[columnas_originales]
    .fillna("")
    .apply(lambda fila: fila.str.strip().eq("").all(), axis=1)
)

resumen_dimension = pd.DataFrame({
    "métrica": [
        "Registros crudos",
        "Registros no vacíos",
        "Filas completamente vacías",
        "Variables originales",
        "Columnas de trazabilidad",
    ],
    "valor": [
        len(df_raw),
        (~filas_completamente_vacias).sum(),
        filas_completamente_vacias.sum(),
        len(columnas_originales),
        2,
    ],
})

display(resumen_dimension)

### Análisis 3.a

> Escriba aquí su interpretación.

## 3.b Tipo de dato de cada variable

In [ ]:
tipos = pd.DataFrame({
    "variable": df_raw.columns,
    "tipo_leído": df_raw.dtypes.astype(str).values,
})

display(tipos)

### Análisis 3.b

Recuerde que códigos, distritos y teléfonos deben conservarse como texto,
aunque contengan números.

## 3.c Cantidad y porcentaje de valores faltantes

In [ ]:
marcadores_faltantes = {
    "", "N/A", "NA", "NULL", "-", ".", "SIN DATO",
    "S/D", "SD", "NO APLICA", "N.A."
}

faltantes = []

for columna in columnas_originales:
    normalizada = (
        df_raw[columna]
        .fillna("")
        .astype("string")
        .str.strip()
        .str.upper()
    )

    mascara = normalizada.isin(marcadores_faltantes)

    faltantes.append({
        "variable": columna,
        "faltantes": int(mascara.sum()),
        "porcentaje": round(mascara.mean() * 100, 4),
    })

tabla_faltantes = (
    pd.DataFrame(faltantes)
    .sort_values("faltantes", ascending=False)
)

display(tabla_faltantes)

### Análisis 3.c

> Escriba aquí las variables más afectadas.

## 3.d Cantidad de valores únicos

In [ ]:
tabla_unicos = pd.DataFrame({
    "variable": columnas_originales,
    "valores_unicos": [
        df_raw[columna].nunique(dropna=False)
        for columna in columnas_originales
    ],
}).sort_values("valores_unicos", ascending=False)

display(tabla_unicos)

### Análisis 3.d

> Escriba aquí su interpretación.

## 3.e Registros duplicados exactos

In [ ]:
duplicados_exactos = df_raw.duplicated(
    subset=columnas_originales,
    keep=False,
)

print(
    "Filas pertenecientes a grupos duplicados:",
    int(duplicados_exactos.sum()),
)

display(
    df_raw.loc[
        duplicados_exactos,
        columnas_originales + ["archivo_origen", "fila_origen"],
    ].sort_values(["CODIGO", "archivo_origen"])
)

### Análisis 3.e

Verifique si los duplicados corresponden a registros reales o a filas
completamente vacías. No elimine todavía ningún registro.

## 3.f Valores fuera de dominio o inconsistentes

In [ ]:
variables_categoricas = [
    "DEPARTAMENTO", "NIVEL", "SECTOR", "AREA",
    "STATUS", "MODALIDAD", "JORNADA", "PLAN",
    "DEPARTAMENTAL",
]

for variable in variables_categoricas:
    print(f"\n{variable}")
    display(
        df_raw[variable]
        .fillna("")
        .value_counts(dropna=False)
        .rename_axis(variable)
        .reset_index(name="cantidad")
    )

### Análisis 3.f

> Documente dominios y categorías por revisar.

## 3.g Variables con formatos inconsistentes

In [ ]:
diagnostico_formatos = []

for variable in columnas_originales:
    texto = df_raw[variable].fillna("").astype("string")

    diagnostico_formatos.append({
        "variable": variable,
        "espacios_inicio_fin": int(
            texto.ne(texto.str.strip()).sum()
        ),
        "espacios_multiples": int(
            texto.str.contains(r"\s{2,}", regex=True).sum()
        ),
        "punto_final": int(
            texto.str.strip().str.endswith(".").sum()
        ),
    })

tabla_formatos = pd.DataFrame(diagnostico_formatos)
display(
    tabla_formatos.loc[
        tabla_formatos[
            ["espacios_inicio_fin", "espacios_multiples", "punto_final"]
        ].sum(axis=1) > 0
    ]
)

telefono = df_raw["TELEFONO"].fillna("").str.strip()

resumen_telefono = pd.DataFrame({
    "métrica": [
        "Vacíos",
        "Exactamente 8 dígitos",
        "Formato diferente de 8 dígitos",
        "Con letras o texto adicional",
    ],
    "cantidad": [
        telefono.eq("").sum(),
        telefono.str.fullmatch(r"\d{8}").sum(),
        (
            telefono.ne("")
            & ~telefono.str.fullmatch(r"\d{8}")
        ).sum(),
        telefono.str.contains(
            r"[A-Za-zÁÉÍÓÚÑáéíóúñ]",
            regex=True,
        ).sum(),
    ],
})

display(resumen_telefono)

### Análisis 3.g

> Escriba aquí los formatos inconsistentes.

## 3.h Problemas potenciales de calidad

In [ ]:
problemas = pd.read_csv(
    REPORT_DIR / "problemas_calidad_preliminares.csv",
    encoding="utf-8-sig",
)

display(problemas)

### Análisis 3.h

Resuma los principales riesgos detectados. Todavía no proponga cambios
definitivos; las reglas deberán formalizarse en el plan de limpieza.